<a href="https://colab.research.google.com/github/KeerthanaSistla/Natural-Language-Processing/blob/main/NLP_Assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1. Machine Translation using LSTM (English → French)

In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

input_texts = ["hello", "how are you", "good morning"]
target_texts = ["<start> bonjour <end>", "<start> comment ca va <end>", "<start> bonjour <end>"]

tokenizer_in = Tokenizer(filters='')
tokenizer_in.fit_on_texts(input_texts)
encoder_input = pad_sequences(tokenizer_in.texts_to_sequences(input_texts), padding='post')

tokenizer_out = Tokenizer(filters='')
tokenizer_out.fit_on_texts(target_texts)
decoder_input = pad_sequences(tokenizer_out.texts_to_sequences(target_texts), padding='post')

decoder_target = np.zeros_like(decoder_input)
for i in range(decoder_input.shape[0]):
    decoder_target[i, :-1] = decoder_input[i, 1:]
decoder_target = np.expand_dims(decoder_target, -1)

vocab_in = len(tokenizer_in.word_index) + 1
vocab_out = len(tokenizer_out.word_index) + 1

latent_dim = 128

encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(vocab_in, 64)(encoder_inputs)
_, state_h, state_c = LSTM(latent_dim, return_state=True)(enc_emb)

decoder_inputs = Input(shape=(None,))
dec_emb_layer = Embedding(vocab_out, 64)
dec_emb = dec_emb_layer(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=[state_h, state_c])

decoder_dense = Dense(vocab_out, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
model.fit([encoder_input, decoder_input], decoder_target, epochs=200, verbose=0)

encoder_model = Model(encoder_inputs, [state_h, state_c])

decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
dec_emb2 = dec_emb_layer(decoder_inputs)
decoder_outputs2, h, c = decoder_lstm(dec_emb2, initial_state=[decoder_state_input_h, decoder_state_input_c])
decoder_outputs2 = decoder_dense(decoder_outputs2)

decoder_model = Model([decoder_inputs, decoder_state_input_h, decoder_state_input_c],
                      [decoder_outputs2, h, c])

reverse_target_index = {i: w for w, i in tokenizer_out.word_index.items()}
reverse_target_index[0] = ""

start_token = tokenizer_out.word_index['<start>']
end_token = tokenizer_out.word_index['<end>']

def translate(sentence):
    seq = pad_sequences(tokenizer_in.texts_to_sequences([sentence]), maxlen=encoder_input.shape[1], padding='post')
    states = encoder_model.predict(seq, verbose=0)
    target_seq = np.array([[start_token]])
    result = []

    for _ in range(10):
        output, h, c = decoder_model.predict([target_seq] + states, verbose=0)
        idx = np.argmax(output[0, -1, :])
        word = reverse_target_index.get(idx, "")
        if word == "<end>" or word == "":
            break
        result.append(word)
        target_seq = np.array([[idx]])
        states = [h, c]
    return " ".join(result)

for s in input_texts:
    print(s, "->", translate(s))

hello -> bonjour
how are you -> comment ca va
good morning -> bonjour
